# DistilBERT — a distilled version of BERT: smaller, faster, cheaper and lighter (2019)

_Papers · Transformers & LLMs_

**Train a small student to copy a big teacher's soft answers, keeping most of the accuracy at a fraction of the cost.**

---

This notebook is a practice scaffold for the **DistilBERT — a distilled version of BERT: smaller, faster, cheaper and lighter (2019)** lesson. We build the distillation recipe one piece at a time: run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Sanity-check the distillation loss by hand

Distillation trains a student to match a teacher's **softened** output distribution. We soften logits by dividing by a temperature `T` before the softmax, then measure how far the student is from the teacher with the cross-entropy `L_ce = -Σ tᵢ log sᵢ`. Working one example through first makes the loss concrete before we wire it into a training loop.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)
K = 4   # number of classes

# Sanity-check the worked example: softmax-temperature + L_ce, T = 2.
T = 2.0
zt = torch.tensor([2.0, 1.0, 0.1])          # teacher logits
zs = torch.tensor([1.5, 0.5, 0.2])          # student logits

t = F.softmax(zt / T, dim=0)                 # teacher soft targets
s = F.softmax(zs / T, dim=0)                 # student soft predictions
L_ce = -(t * torch.log(s)).sum()            # distillation loss  -sum_i t_i log s_i

print("teacher soft t =", [round(v, 4) for v in t.tolist()])   # [0.5017, 0.3043, 0.194]
print("student soft s =", [round(v, 4) for v in s.tolist()])   # [0.4698, 0.2849, 0.2453]
print("L_ce =", round(L_ce.item(), 4))                          # 1.0337

### Step 2 — Build the data and the two networks

We make a toy 4-class problem: four Gaussian blobs arranged on a ring. The **teacher** gets plenty of data (2000 points); the **student** is deliberately data-starved (only 60 labels) — exactly the setting where the teacher's extra knowledge should help. The teacher is a wide MLP, while the student is a tiny one, so any accuracy gain can't come from extra capacity.

In [ ]:
# A toy 4-class dataset: four Gaussian blobs on a ring.
def make_data(n, seed, spread=1.0):
    g = torch.Generator().manual_seed(seed)
    y = torch.randint(0, K, (n,), generator=g)
    centers = torch.tensor([[2.0 * math.cos(2 * math.pi * k / K),
                             2.0 * math.sin(2 * math.pi * k / K)] for k in range(K)])
    X = centers[y] + torch.randn(n, 2, generator=g) * spread
    return X, y

Xbig, ybig = make_data(2000, 99)   # the teacher trains on plenty of data
Xtr,  ytr  = make_data(60,  1)     # the student is DATA-STARVED: only 60 labels
Xte,  yte  = make_data(800, 51)    # held-out test set


# Teacher (big) and Student (tiny) MLPs.
def teacher_mlp():
    return nn.Sequential(nn.Linear(2, 128), nn.ReLU(),
                         nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, K))

def student_mlp(h):
    return nn.Sequential(nn.Linear(2, h), nn.ReLU(), nn.Linear(h, K))   # tiny

def accuracy(net, X, y):
    net.eval()
    with torch.no_grad():
        return round((net(X).argmax(1) == y).float().mean().item(), 4)

def train_hard(net, X, y, epochs=400, lr=0.02):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    net.train()
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.cross_entropy(net(X), y)
        loss.backward()
        opt.step()
    return net

### Step 3 — Train the teacher on the large dataset

The teacher learns on all 2000 points with ordinary hard-label cross-entropy. Its test accuracy is the bar the tiny student will try to reach. Once trained, the teacher is **frozen** — from here on we only read its soft predictions.

In [ ]:
# Train the TEACHER (big) on the large dataset.
torch.manual_seed(0)
teacher = train_hard(teacher_mlp(), Xbig, ybig, epochs=500)
teacher.eval()
print("teacher test acc        :", accuracy(teacher, Xte, yte))   # 0.8288

### Step 4 — Distill the tiny student

Now the payoff. The student trains on the same 60 labels, but its loss blends two terms: `alpha * T² * L_ce` (match the teacher's soft targets) plus `(1 - alpha) * L_hard` (the true labels, playing the role of BERT's masked-LM loss). The `T²` factor rescales the softened gradient back up so the soft term keeps its weight; the soft targets carry the teacher's "dark knowledge" about how classes relate.

In [ ]:
# DISTILL a tiny student: L = alpha*T^2*L_ce + (1-alpha)*L_hard.
def distill(student, T=3.0, alpha=0.8, epochs=400, lr=0.02):
    with torch.no_grad():                                  # teacher is FROZEN
        t_soft = F.softmax(teacher(Xtr) / T, dim=1)        # soft targets (constants)
    opt = torch.optim.Adam(student.parameters(), lr=lr)
    student.train()
    for _ in range(epochs):
        opt.zero_grad()
        logits = student(Xtr)
        log_s  = F.log_softmax(logits / T, dim=1)          # student softened
        L_ce   = -(t_soft * log_s).sum(dim=1).mean()       # -sum_i t_i log s_i
        L_hard = F.cross_entropy(logits, ytr)              # the L_mlm role
        loss   = alpha * (T * T) * L_ce + (1 - alpha) * L_hard
        loss.backward()
        opt.step()
    return student

torch.manual_seed(0)
student_distill = distill(student_mlp(3))                 # tiny student, DISTILLED
print("student (distilled) acc :", accuracy(student_distill, Xte, yte))   # 0.8425

### Step 5 — Ablation: the same student, no teacher

To prove the gain comes from distillation and not from the architecture, we train an identical tiny student on the same 60 labels with hard-label cross-entropy only. Comparing this from-scratch student to the distilled one isolates the soft targets as the cause of any difference.

In [ ]:
# ABLATION: same tiny student, those 60 hard labels only (from scratch).
torch.manual_seed(0)
student_scratch = train_hard(student_mlp(3), Xtr, ytr)   # SAME size, no teacher
print("student (scratch)   acc :", accuracy(student_scratch, Xte, yte))   # 0.7937

# Our run: teacher 0.8288, distilled student 0.8425, from-scratch student 0.7937.
# Same tiny student + same 60 labels; the distilled one wins by ~5 points and nearly
# matches the teacher -- the teacher's soft targets supply what the few labels cannot.
# (This is our small run, not the paper's reported numbers.)

## Visualize the data & results

_At equal student size, does distillation (matching the teacher's soft targets) beat training the same small student from scratch on hard labels?_

### Step 6 — Rebuild the experiment self-contained

This visualization cell stands on its own, so it re-imports and re-creates the data, the teacher, and the student helpers exactly as above. Re-running the whole pipeline here keeps the panel reproducible in isolation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

K = 4

def make_data(n, seed, spread=1.0):
    g = torch.Generator().manual_seed(seed)
    y = torch.randint(0, K, (n,), generator=g)
    centers = torch.tensor([[2.0 * math.cos(2 * math.pi * k / K),
                             2.0 * math.sin(2 * math.pi * k / K)]
                            for k in range(K)])
    return centers[y] + torch.randn(n, 2, generator=g) * spread, y

Xbig, ybig = make_data(2000, 99)   # teacher: lots of data
Xtr,  ytr  = make_data(60,   1)    # student: only 60 labels (data-starved)
Xte,  yte  = make_data(800, 51)

def teacher_mlp():
    return nn.Sequential(nn.Linear(2, 128), nn.ReLU(),
                         nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, K))

def student_mlp(h):
    return nn.Sequential(nn.Linear(2, h), nn.ReLU(), nn.Linear(h, K))   # tiny

def acc(net, X, y):
    net.eval()
    with torch.no_grad():
        return round((net(X).argmax(1) == y).float().mean().item(), 4)

def train_hard(net, X, y, epochs=400, lr=0.02):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    net.train()
    for _ in range(epochs):
        opt.zero_grad()
        F.cross_entropy(net(X), y).backward()
        opt.step()
    return net

torch.manual_seed(0)
teacher = train_hard(teacher_mlp(), Xbig, ybig, epochs=500)
teacher.eval()

### Step 7 — Train both students and print the head-to-head

Now train the distilled student and the from-scratch student under identical seeds, then print all three accuracies side by side. The distilled student should land above the from-scratch one and close to the teacher — the headline result, distillation beating same-size training on hard labels alone.

In [ ]:
def distill(student, T=3.0, alpha=0.8, epochs=400, lr=0.02):
    with torch.no_grad():
        t_soft = F.softmax(teacher(Xtr) / T, dim=1)
    opt = torch.optim.Adam(student.parameters(), lr=lr)
    student.train()
    for _ in range(epochs):
        opt.zero_grad()
        z = student(Xtr)
        L_ce = -(t_soft * F.log_softmax(z / T, dim=1)).sum(1).mean()
        loss = alpha * (T * T) * L_ce + (1 - alpha) * F.cross_entropy(z, ytr)
        loss.backward()
        opt.step()
    return student

torch.manual_seed(0)
s_distill = distill(student_mlp(3))
torch.manual_seed(0)
s_scratch = train_hard(student_mlp(3), Xtr, ytr)

print("Teacher                 :", acc(teacher,   Xte, yte))   # 0.8288
print("Student distilled (h3)  :", acc(s_distill, Xte, yte))   # 0.8425
print("Student scratch   (h3)  :", acc(s_scratch, Xte, yte))   # 0.7937

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation (distill vs from-scratch). You have a trained teacher and a small student.
            Train the student two ways &mdash; (A) with the distillation loss matching the teacher's soft
            targets, (B) on the hard true labels only &mdash; keeping the student size, data, and optimizer
            identical. Which reaches higher test accuracy, and what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Train student A with $L = \alpha T^2 L_{ce} + (1-\alpha) L_{\text{hard}}$ using the frozen teacher's softened targets. — _This is the distillation recipe: the student copies the teacher's full soft distribution, inheriting its similarity structure._
- Train student B with hard-label cross-entropy only &mdash; same architecture, data, epochs, seed. — _Changing only the loss isolates distillation as the cause of any accuracy difference._
- Compare final test accuracy of A vs B. — _If A > B at equal size, the gain comes from the teacher's dark knowledge, not extra capacity._

**Answer:** The distilled student (A) typically reaches higher test accuracy than the matched
                 from-scratch student (B). Since the two students are identical in size and see the same data,
                 the gain is attributable to the soft targets: the teacher's full distribution carries
                 the similarity ("dark knowledge") information a one-hot label discards. This is the toy
                 version of DistilBERT retaining ~97% of BERT's GLUE score at 40% smaller. The CODEVIZ panel
                 shows exactly this contrast (our small run, not the paper's numbers).

</details>

**Problem 2.** Your worked example used $T = 2$ and got teacher soft targets $t = [0.5017, 0.3043, 0.1940]$ from
            logits $[2.0, 1.0, 0.1]$. Recompute the teacher distribution at $T = 1$. What happens to the
            third class, and why does the paper want a higher $T$ during training?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- At $T = 1$, exponentiate the raw logits: $[e^{2.0}, e^{1.0}, e^{0.1}] = [7.389, 2.718, 1.105]$, sum $= 11.213$. — _$T = 1$ is the ordinary softmax &mdash; no softening._
- Normalize: $t = [0.6590, 0.2424, 0.0986]$. — _The distribution is peakier than at $T = 2$: more mass on class 1, less on class 3._
- Compare the third class: $0.0986$ at $T=1$ vs $0.1940$ at $T=2$. — _Raising $T$ lifts the small probabilities, making the runner-up classes (the dark knowledge) visible to the student._

**Answer:** At $T = 1$ the teacher distribution is $[0.659, 0.242, 0.099]$ &mdash; peakier, with the
                 third class squashed to $\approx 0.099$. Raising $T$ to $2$ lifted it to $0.194$. The paper
                 uses $T \gt 1$ at training time precisely so these small "dark knowledge" probabilities are
                 large enough to give the student a useful gradient; $T$ is reset to $1$ at inference.

</details>

**Problem 3.** You implement the distillation loss but forget the $T^2$ factor and use $T = 4$. You find
            distillation barely beats the from-scratch baseline. What went wrong, and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that softening by $T$ shrinks the soft-cross-entropy gradient by a factor of about $1/T^2$. — _Both the student softmax and the teacher targets are flattened by $T$, scaling the gradient down quadratically._
- At $T = 4$ the soft term's gradient is ~$1/16$ of its un-scaled magnitude, so when mixed with the hard loss it is almost ignored. — _The optimizer effectively trains on the hard labels alone &mdash; little better than the baseline._
- Multiply $L_{ce}$ by $T^2 = 16$ before combining with the hard loss. — _This restores the soft gradient to a comparable scale (Hinton et al. 2015), so distillation contributes properly._

**Answer:** Softening by $T$ reduces the soft-loss gradient by ~$1/T^2$, so at $T = 4$ the soft term is
                 ~16&times; too weak and the student essentially trains on hard labels only. The fix is to
                 multiply the soft (distillation) loss by $T^2$ before mixing it with the hard loss, restoring
                 the gradient scale &mdash; the standard Hinton-et-al. correction.

</details>